# Модуль 2. Углублённое ООП на Python: протоколы, dataclasses, ABC

---

## О чём этот модуль

В первом модуле мы освоили базовые принципы ООП: классы, объекты, инкапсуляцию, наследование, полиморфизм. Этого достаточно для небольших проектов, но в реальной разработке игр архитектура быстро усложняется. Возникают вопросы:

- Как гарантировать, что все подклассы реализуют нужные методы?
- Как описать «контракт» класса, не привязываясь к конкретной реализации?
- Как избежать дублирования кода при создании множества классов с похожими полями?
- Как оптимизировать память, когда в игре одновременно существуют тысячи объектов?
- Как выбирать между наследованием и композицией?

Этот модуль отвечает на эти вопросы. Мы изучим **современные инструменты Python**, которые превращают ООП из базовой техники в полноценный архитектурный подход:

- **ABC** — абстрактные базовые классы для явного описания интерфейсов.
- **dataclasses** — автоматическая генерация методов для классов-данных.
- **Протоколы** — структурная типизация без наследования.
- **Миксины** — переиспользование поведения через множественное наследование.
- **`__slots__`** — оптимизация памяти для больших коллекций объектов.
- **Дескрипторы** — управление доступом к атрибутам на низком уровне.
- **Композиция vs наследование** — осознанный выбор архитектурного подхода.

Все эти инструменты напрямую применяются в игровой разработке: компоненты, конфигурации, интерфейсы оружия, оптимизация партиклов и снарядов.

---

## Содержание
1. Абстрактные базовые классы (`abc.ABC`, `@abstractmethod`)
2. `dataclasses`: автоматизация классов данных
3. Протоколы (`typing.Protocol`): структурная типизация
4. Миксины: переиспользование поведения
5. Композиция vs наследование
6. `__slots__`: оптимизация памяти
7. Дескрипторы: управление атрибутами
8. Современные практики проектирования интерфейсов
9. Итоги и литература

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# 1. Абстрактные базовые классы (`abc.ABC`, `@abstractmethod`)

## 1.1. Проблема: как гарантировать реализацию методов

Представьте, что вы проектируете систему оружия в игре. У вас есть базовый класс `Weapon` и подклассы `Sword`, `Bow`, `Staff`. Каждое оружие должно уметь атаковать, но **как именно** — дело конкретного класса.

Если просто определить метод `attack()` в базовом классе с пустым телом, возникает проблема: подкласс может **забыть** переопределить метод. Игра скомпилируется, но при вызове `attack()` произойдёт нечто неожиданное (или ничего не произойдёт).

```python
# ПЛОХО: нет гарантии, что подкласс реализует attack
class Weapon:
    def attack(self, target):
        pass  # что делать?
```

**Абстрактный базовый класс (ABC)** решает эту проблему: он **запрещает** создавать экземпляры класса, пока все абстрактные методы не будут реализованы в подклассе. Это как «контракт», который подкласс обязан выполнить.

## 1.2. Модуль `abc`

Модуль `abc` предоставляет инфраструктуру для определения абстрактных базовых классов[reference:0]. Ключевые элементы:

- **`abc.ABC`** — базовый класс для создания ABC.
- **`@abstractmethod`** — декоратор, помечающий метод как абстрактный.
- **`abc.ABCMeta`** — метакласс, лежащий в основе `ABC`.

## 1.3. Простой пример

```python
from abc import ABC, abstractmethod

class Weapon(ABC):
    """Абстрактное оружие."""

    def __init__(self, damage):
        self.damage = damage

    @abstractmethod
    def attack(self, target):
        """Атаковать цель. Каждый подкласс реализует по-своему."""
        ...

    @abstractmethod
    def get_description(self):
        """Вернуть описание оружия для UI."""
        ...

    def show_info(self):
        """Конкретный метод — доступен всем подклассам."""
        print(f"{self.get_description()} (damage: {self.damage})")
```

**Что произойдёт, если попытаться создать `Weapon`?**

```python
w = Weapon(10)  # TypeError: Can't instantiate abstract class Weapon
```

Python не позволит создать экземпляр, пока не будут реализованы **все** абстрактные методы[reference:1].

## 1.4. Реализация подкласса

```python
class Sword(Weapon):
    def attack(self, target):
        print(f"Sword swings at {target.name} for {self.damage} damage!")
        target.take_damage(self.damage)

    def get_description(self):
        return "A sharp sword"


class Bow(Weapon):
    def attack(self, target):
        print(f"Bow shoots an arrow at {target.name} for {self.damage} damage!")
        target.take_damage(self.damage)

    def get_description(self):
        return "A longbow"


sword = Sword(15)
bow = Bow(20)
sword.show_info()   # A sharp sword (damage: 15)
bow.show_info()     # A longbow (damage: 20)
```

## 1.5. Виртуальные подклассы

ABC позволяет регистрировать **виртуальные подклассы** — классы, которые не наследуются от ABC, но считаются его подклассами при проверке `isinstance()` и `issubclass()`[reference:2]:

```python
from abc import ABC, abstractmethod

class Movable(ABC):
    @abstractmethod
    def move(self, dx, dy):
        ...

class Vector2D:
    """Не наследуется от Movable, но реализует move."""
    def __init__(self, x, y):
        self.x, self.y = x, y

    def move(self, dx, dy):
        self.x += dx
        self.y += dy

# Регистрируем Vector2D как виртуальный подкласс Movable
Movable.register(Vector2D)

v = Vector2D(0, 0)
print(isinstance(v, Movable))  # True
print(issubclass(Vector2D, Movable))  # True
```

**Когда это полезно:** когда вы не контролируете определение класса (например, он из сторонней библиотеки), но хотите, чтобы он считался частью вашей иерархии.

## 1.6. Применение в игровой разработке

- **Базовые классы для оружия, врагов, способностей.** ABC гарантирует, что каждый подкласс реализует `attack()`, `update()`, `draw()`.
- **Интерфейсы для систем.** Например, `Renderable`, `PhysicsBody`, `Saveable`.
- **Плагины и расширения.** Сторонние разработчики могут реализовать ваш ABC.

**Для углублённого изучения:**
- [Real Python: abc Module](https://realpython.com/ref/stdlib/abc/)
- [Python docs: abc — Abstract Base Classes](https://docs.python.org/3/library/abc.html)
- [PEP 3119 — Introducing Abstract Base Classes](https://peps.python.org/pep-3119/)

In [ ]:
from abc import ABC, abstractmethod


class Weapon(ABC):
    def __init__(self, damage):
        self.damage = damage

    @abstractmethod
    def attack(self, target):
        ...

    @abstractmethod
    def get_description(self):
        ...

    def show_info(self):
        print(f"{self.get_description()} (damage: {self.damage})")


class Sword(Weapon):
    def attack(self, target):
        print(f"Sword hits {target} for {self.damage}")

    def get_description(self):
        return "A sharp sword"


class Bow(Weapon):
    def attack(self, target):
        print(f"Bow shoots {target} for {self.damage}")

    def get_description(self):
        return "A longbow"


# Попытка создать абстрактный класс
try:
    Weapon(10)
except TypeError as e:
    print(f"Ошибка: {e}")

# Создание конкретных подклассов
sword = Sword(15)
bow = Bow(20)
sword.show_info()
bow.show_info()

# Проверка isinstance
print(f"Sword is Weapon: {isinstance(sword, Weapon)}")
print(f"Bow is Weapon: {isinstance(bow, Weapon)}")

---
# 2. `dataclasses`: автоматизация классов данных

## 2.1. Проблема: много шаблонного кода

В игровой разработке часто встречаются классы, которые содержат только данные и почти не имеют поведения: конфигурации, компоненты, структуры для сохранения, DTO. Писать для них `__init__`, `__repr__`, `__eq__` вручную — утомительно и чревато ошибками.

```python
# Вручную — много кода, легко ошибиться
class Vector2:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector2(x={self.x}, y={self.y})"

    def __eq__(self, other):
        if not isinstance(other, Vector2):
            return NotImplemented
        return self.x == other.x and self.y == other.y
```

## 2.2. Решение: `@dataclass`

Декоратор `@dataclass` автоматически генерирует специальные методы на основе **аннотаций типов**[reference:3]:

```python
from dataclasses import dataclass

@dataclass
class Vector2:
    x: float
    y: float
```

**Что генерируется автоматически:**
- `__init__(self, x, y)` — конструктор[reference:4].
- `__repr__(self)` — читаемое представление[reference:5].
- `__eq__(self, other)` — сравнение по полям.

```python
v1 = Vector2(1.0, 2.0)
v2 = Vector2(1.0, 2.0)
print(v1)           # Vector2(x=1.0, y=2.0)
print(v1 == v2)     # True
```

## 2.3. Параметры `@dataclass`

Декоратор принимает несколько параметров[reference:6]:

| Параметр | По умолчанию | Что делает |
|---|---|---|
| `init` | `True` | Генерировать `__init__` |
| `repr` | `True` | Генерировать `__repr__` |
| `eq` | `True` | Генерировать `__eq__` |
| `order` | `False` | Генерировать `__lt__`, `__gt__` и др. |
| `frozen` | `False` | Сделать экземпляры неизменяемыми |
| `slots` | `False` | Добавить `__slots__` (Python 3.10+) |

## 2.4. `frozen=True`: неизменяемые объекты

Если объект не должен меняться после создания, используйте `frozen=True`[reference:7]:

```python
@dataclass(frozen=True)
class Config:
    width: int
    height: int
    title: str = "Game"

cfg = Config(800, 600)
# cfg.width = 1024  # dataclasses.FrozenInstanceError
```

**Зачем это в играх:**
- **Конфигурации** не должны меняться во время выполнения.
- **Ключи кэша** (например, для ассетов) должны быть хешируемыми и неизменяемыми.
- **События** в event-системах — иммутабельны по природе.

## 2.5. `field()`: тонкая настройка полей

Функция `field()` позволяет настроить поведение отдельных полей:

```python
from dataclasses import dataclass, field

@dataclass
class Inventory:
    owner: str
    items: list = field(default_factory=list)  # новый список для каждого объекта
    gold: int = 0
    _internal_id: int = field(default=0, repr=False)  # не показывать в repr

inv1 = Inventory("Hero")
inv2 = Inventory("Mage")
inv1.items.append("sword")
print(inv1.items)  # ['sword']
print(inv2.items)  # [] — независимый список
```

**Ключевой момент:** `default_factory` решает проблему изменяемых значений по умолчанию, которую мы обсуждали в модуле 1.

## 2.6. Применение в игровой разработке

- **Компоненты:** `Position(x, y)`, `Health(current, max)`, `Velocity(dx, dy)`.
- **Конфигурации:** параметры уровня, настройки оружия, таблицы опыта.
- **События:** `DamageEvent(source, target, amount)`.
- **Сохранения:** `SaveData(player_pos, inventory, quests)`.

**Для углублённого изучения:**
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)
- [PEP 557 — Data Classes](https://peps.python.org/pep-0557/)
- [Real Python: Data Classes in Python 3.7+](https://realpython.com/python-data-classes/)

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Vector2:
    x: float
    y: float

    def length(self) -> float:
        return (self.x ** 2 + self.y ** 2) ** 0.5


v1 = Vector2(3.0, 4.0)
v2 = Vector2(3.0, 4.0)
print(v1)
print(f"Length: {v1.length()}")
print(f"v1 == v2: {v1 == v2}")


@dataclass(frozen=True)
class GameConfig:
    width: int
    height: int
    title: str = "My Game"
    fps: int = 60


cfg = GameConfig(1920, 1080, "Space Shooter")
print(cfg)


@dataclass
class Inventory:
    owner: str
    items: list = field(default_factory=list)
    gold: int = 0


inv1 = Inventory("Hero")
inv2 = Inventory("Mage")
inv1.items.append("sword")
print(f"inv1: {inv1.items}, inv2: {inv2.items}")

---
# 3. Протоколы (`typing.Protocol`): структурная типизация

## 3.1. Проблема: наследование не всегда уместно

В модуле 1 мы обсуждали «утиную типизацию»: если объект имеет нужный метод, Python не проверяет его тип. Но статические анализаторы (mypy, PyCharm) **не понимают** утиную типизацию без подсказок.

**Проблема:** как описать «объект, у которого есть метод `draw()`», не заставляя все классы наследоваться от общего базового класса?

**Решение:** протоколы (`typing.Protocol`).

## 3.2. Что такое протокол

**Протокол** — это тип, поддерживающий **структурную типизацию** (structural subtyping)[reference:8]. Это означает: класс совместим с протоколом, если он **имеет** нужные методы и атрибуты, а не если он **наследуется** от протокола.

**Аналогия:** в Python есть «протокол итератора» — любой объект с методами `__iter__` и `__next__` является итератором. Не нужно ни от чего наследоваться. `typing.Protocol` позволяет описывать такие неформальные протоколы **явно**.

## 3.3. Определение протокола

```python
from typing import Protocol

class Drawable(Protocol):
    """Всё, что можно нарисовать."""
    def draw(self, screen) -> None:
        ...

class Updatable(Protocol):
    """Всё, что можно обновить."""
    def update(self, dt: float) -> None:
        ...
```

**Обратите внимание:** тело методов — `...` (многоточие). Протокол описывает только **сигнатуру**, не реализацию.

## 3.4. Использование протокола

Любой класс, имеющий метод `draw()`, совместим с `Drawable` — даже если он не наследуется от него[reference:9]:

```python
class Player:
    def draw(self, screen):
        print(f"Drawing player on {screen}")

class Enemy:
    def draw(self, screen):
        print(f"Drawing enemy on {screen}")

def render_all(objects: list[Drawable], screen):
    """Принимает любые объекты, реализующие Drawable."""
    for obj in objects:
        obj.draw(screen)

render_all([Player(), Enemy()], "main_screen")
```

**Статический анализатор** (mypy) проверит, что `Player` и `Enemy` имеют метод `draw()`, даже если они не наследуются от `Drawable`.

## 3.5. `@runtime_checkable`

По умолчанию протоколы работают только на уровне типов (статически). Чтобы использовать `isinstance()` с протоколом, нужно добавить декоратор `@runtime_checkable`[reference:10]:

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class Drawable(Protocol):
    def draw(self, screen) -> None:
        ...

print(isinstance(Player(), Drawable))  # True
print(isinstance(42, Drawable))        # False
```

**Важно:** `runtime_checkable` проверяет только **наличие** методов, но не их сигнатуры. Это «грубая» проверка.

## 3.6. Протоколы vs ABC

| Критерий | ABC | Protocol |
|---|---|---|
| Требует наследования | Да | Нет |
| Проверка во время выполнения | Да | Только с `@runtime_checkable` |
| Статическая проверка | Да | Да |
| Когда использовать | Когда нужен явный контракт | Когда важно «что умеет», а не «чем является» |

**Правило выбора:** ABC — когда вы **владеете** иерархией и хотите её контролировать. Протоколы — когда вы хотите работать с **любыми** объектами, имеющими нужные методы.

## 3.7. Применение в игровой разработке

- **Системы рендеринга:** `Drawable`, `Renderable`.
- **Системы обновления:** `Updatable`, `Tickable`.
- **Системы сохранения:** `Serializable`, `Saveable`.
- **Обработка событий:** `EventListener`, `Observer`.

**Для углублённого изучения:**
- [Python docs: Protocols](https://typing.python.org/en/latest/spec/protocol.html)
- [PEP 544 — Protocols: Structural subtyping](https://peps.python.org/pep-0544/)
- [Real Python: Structural Subtyping with Protocols](https://realpython.com/python-protocol/)

In [ ]:
from typing import Protocol, runtime_checkable


class Drawable(Protocol):
    def draw(self, screen: str) -> None:
        ...


@runtime_checkable
class Updatable(Protocol):
    def update(self, dt: float) -> None:
        ...


class Player:
    def draw(self, screen: str) -> None:
        print(f"Player drawn on {screen}")

    def update(self, dt: float) -> None:
        print(f"Player updated by {dt}s")


class Enemy:
    def draw(self, screen: str) -> None:
        print(f"Enemy drawn on {screen}")


def render_all(objects, screen: str) -> None:
    for obj in objects:
        obj.draw(screen)


render_all([Player(), Enemy()], "main_screen")

print(f"Player is Drawable: {isinstance(Player(), Drawable)}")
print(f"Player is Updatable: {isinstance(Player(), Updatable)}")
print(f"Enemy is Updatable: {isinstance(Enemy(), Updatable)}")

---
# 4. Миксины: переиспользование поведения

## 4.1. Что такое миксин

**Миксин** — это класс, предназначенный для добавления конкретной функциональности другим классам через множественное наследование[reference:11]. Миксины:

- **не предназначены для самостоятельного использования** (нельзя создать экземпляр);
- **не имеют собственного состояния** (`__init__`, атрибуты экземпляра);
- обычно определяют один-два метода, которые «примешиваются» к другим классам.

**Аналогия:** миксин — это как «приправа» к блюду. Сама по себе приправа несъедобна, но добавляет вкус основному блюду.

## 4.2. Простой миксин

```python
class LoggingMixin:
    """Миксин: добавляет логирование вызовов методов."""
    __slots__ = ()  # у миксина нет собственного состояния

    def log(self, message):
        print(f"[{self.__class__.__name__}] {message}")


class Player(LoggingMixin):
    def __init__(self, name):
        self.name = name

    def attack(self):
        self.log(f"{self.name} attacks!")


p = Player("Hero")
p.attack()  # [Player] Hero attacks!
```

## 4.3. Миксины в стандартной библиотеке

Python Cookbook приводит пример миксинов для расширения словарей[reference:12]:

```python
class LoggedMappingMixin:
    """Добавляет логирование к операциям get/set/delete."""
    __slots__ = ()

    def __getitem__(self, key):
        print(f"Getting {key}")
        return super().__getitem__(key)

    def __setitem__(self, key, value):
        print(f"Setting {key} = {value!r}")
        return super().__setitem__(key, value)


class LoggedDict(LoggedMappingMixin, dict):
    pass


d = LoggedDict()
d['x'] = 23      # Setting x = 23
print(d['x'])    # Getting x -> 23
```

**Ключевая идея:** миксин вызывает `super().__setitem__()`, передавая управление следующему классу в MRO. Это называется **кооперативным наследованием**.

## 4.4. Правила создания миксинов

1. **Не иметь `__init__`** — миксин не должен требовать инициализации.
2. **Не иметь состояния** — только методы. Если нужны «пустые» слоты, используйте `__slots__ = ()`.
3. **Вызывать `super()`** — для кооперативного наследования.
4. **Имя должно оканчиваться на `Mixin`** — это соглашение, облегчающее чтение.
5. **Быть независимым** — миксин не должен предполагать, с каким классом он будет смешан.

## 4.5. Применение в игровой разработке

- **`SerializableMixin`** — добавляет `to_dict()` и `from_dict()`.
- **`LoggingMixin`** — логирование действий.
- **`EventListenerMixin`** — подписка на события.
- **`PooledMixin`** — для объектов из пула (снаряды, партиклы).

**Для углублённого изучения:**
- [Python Cookbook: Mixins](https://python-cookbook.readthedocs.io/zh-cn/stable/c08/p18_extending_classes_with_mixins.html)
- [Real Python: Multiple Inheritance and Mixins](https://realpython.com/inheritance-composition-python/)

In [ ]:
class LoggingMixin:
    __slots__ = ()

    def log(self, message):
        print(f"[{self.__class__.__name__}] {message}")


class SerializableMixin:
    __slots__ = ()

    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}


class Player(LoggingMixin, SerializableMixin):
    def __init__(self, name, health=100):
        self.name = name
        self.health = health

    def attack(self):
        self.log(f"{self.name} attacks!")


p = Player("Hero", 120)
p.attack()
print(p.to_dict())

---
# 5. Композиция vs наследование

## 5.1. Проблема наследования

Наследование моделирует отношение **«является» (is-a)**: `Dog` является `Animal`. Композиция моделирует **«имеет» (has-a)**: `Car` имеет `Engine`[reference:13].

**Проблема:** наследование вдоль нескольких осей приводит к **комбинаторному взрыву** подклассов. Если у персонажа есть тип движения (3 варианта) и тип оружия (2 варианта), наследование потребует 6 подклассов. Добавьте третий параметр — получите 12[reference:14].

```
Character
├── WalkerWithSword
├── WalkerWithBow
├── FlyerWithSword
├── FlyerWithBow
├── SwimmerWithSword
└── SwimmerWithBow
```

## 5.2. Решение: композиция

**Композиция** — объект **содержит** другие объекты и делегирует им поведение. Вместо 6 подклассов — 3 класса движения + 2 класса оружия + 1 класс персонажа[reference:15].

```python
class WalkMover:
    def move(self, character):
        print(f"{character.name} walks")


class FlyMover:
    def move(self, character):
        print(f"{character.name} flies")


class Sword:
    def attack(self, character, target):
        print(f"{character.name} slashes {target} with sword")


class Bow:
    def attack(self, character, target):
        print(f"{character.name} shoots {target} with bow")


class Character:
    def __init__(self, name, mover, weapon):
        self.name = name
        self.mover = mover      # композиция
        self.weapon = weapon    # композиция

    def move(self):
        self.mover.move(self)

    def attack(self, target):
        self.weapon.attack(self, target)


hero = Character("Hero", FlyMover(), Sword())
hero.move()          # Hero flies
hero.attack("Goblin")  # Hero slashes Goblin with sword
```

## 5.3. Правило выбора

| Критерий | Наследование | Композиция |
|---|---|---|
| Отношение | is-a | has-a |
| Связь | Жёсткая | Слабая |
| Изменение во время выполнения | Невозможно | Возможно |
| Комбинаторный рост | Экспоненциальный | Линейный |
| Когда использовать | Когда отношение стабильно и однозначно | Когда поведение можно комбинировать |

**Правило GoF:** «Предпочитайте композицию наследованию»[reference:16]. Наследование оставьте для случаев, когда отношение действительно «является» и не будет меняться.

**Для углублённого изучения:**
- [CodeGym: Composition vs Inheritance](https://codegym.cc/groups/posts/python-composition-vs-inheritance)
- [Real Python: Inheritance and Composition](https://realpython.com/inheritance-composition-python/)

In [ ]:
class WalkMover:
    def move(self, character):
        print(f"{character.name} walks")


class FlyMover:
    def move(self, character):
        print(f"{character.name} flies")


class Sword:
    def attack(self, character, target):
        print(f"{character.name} slashes {target}")


class Bow:
    def attack(self, character, target):
        print(f"{character.name} shoots {target}")


class Character:
    def __init__(self, name, mover, weapon):
        self.name = name
        self.mover = mover
        self.weapon = weapon

    def move(self):
        self.mover.move(self)

    def attack(self, target):
        self.weapon.attack(self, target)


hero = Character("Hero", FlyMover(), Sword())
hero.move()
hero.attack("Goblin")

# Смена поведения во время выполнения
hero.mover = WalkMover()
hero.weapon = Bow()
hero.move()
hero.attack("Dragon")

---
# 6. `__slots__`: оптимизация памяти

## 6.1. Проблема: словарь у каждого объекта

По умолчанию Python хранит атрибуты экземпляра в **словаре** (`__dict__`). Это даёт гибкость (можно добавлять новые атрибуты в любой момент), но **тратит много памяти**[reference:17].

**Цифры:** каждый экземпляр с 2 атрибутами занимает около **56 байт** только на словарь. Если в игре 100 000 снарядов — это **5.6 МБ только на словари**[reference:18].

## 6.2. Решение: `__slots__`

`__slots__` — список разрешённых атрибутов. Python перестаёт создавать `__dict__` и хранит значения в **фиксированных слотах**[reference:19]:

```python
class Point:
    __slots__ = ('x', 'y')  # только эти атрибуты разрешены

    def __init__(self, x, y):
        self.x = x
        self.y = y


p = Point(1, 2)
# p.z = 3  # AttributeError: 'Point' object has no attribute 'z'
```

## 6.3. Выигрыш

| Метрика | Без `__slots__` | С `__slots__` | Улучшение |
|---|---|---|---|
| Память на 10 000 объектов | ~2.4 МБ | ~0.4 МБ | **80%** |
| Скорость доступа | 0.78 с | 0.52 с | **33%** |

Данные из бенчмарков: `__slots__` снижает память на **30–80%** и ускоряет доступ на **20–50%**[reference:20].

## 6.4. Ограничения

- **Нельзя добавлять новые атрибуты** после создания объекта.
- **Нельзя использовать `__dict__`** (если не добавить `'__dict__'` в слоты явно).
- **Наследование:** если подкласс не определяет `__slots__`, он получит `__dict__` и потеряет выигрыш.

```python
class Base:
    __slots__ = ('a',)

class Child(Base):
    __slots__ = ('b',)  # нужно явно указать
```

## 6.5. Когда использовать в играх

- **Снаряды, частицы, тайлы** — тысячи объектов.
- **Компоненты ECS** — часто создаются и уничтожаются.
- **Сетевые пакеты** — много мелких объектов.

**Не использовать:** для классов с динамическими атрибутами, конфигураций, синглтонов.

**Для углублённого изучения:**
- [Python docs: __slots__](https://docs.python.org/3/reference/datamodel.html#slots)
- [Real Python: __slots__ in Python](https://realpython.com/python-slots/)

In [ ]:
import sys
import tracemalloc


class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class SlottedPoint:
    __slots__ = ('x', 'y')

    def __init__(self, x, y):
        self.x = x
        self.y = y


tracemalloc.start()
regular = [RegularPoint(i, i) for i in range(10_000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Regular: {current / 1024:.1f} KB")

tracemalloc.start()
slotted = [SlottedPoint(i, i) for i in range(10_000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Slotted: {current / 1024:.1f} KB")

print(f"sys.getsizeof(RegularPoint): {sys.getsizeof(regular[0])} bytes")
print(f"sys.getsizeof(SlottedPoint): {sys.getsizeof(slotted[0])} bytes")

---
# 7. Дескрипторы: управление атрибутами

## 7.1. Что такое дескриптор

**Дескриптор** — это объект, который управляет доступом к атрибуту другого объекта. Дескрипторы позволяют настраивать **чтение, запись и удаление** атрибутов[reference:21].

**Аналогия:** дескриптор — это «привратник» у атрибута. Когда кто-то пытается прочитать или записать значение, привратник решает, что делать.

## 7.2. Протокол дескриптора

Дескриптор — это класс, реализующий один или несколько методов:

- `__get__(self, obj, objtype=None)` — чтение атрибута.
- `__set__(self, obj, value)` — запись атрибута.
- `__delete__(self, obj)` — удаление атрибута.

## 7.3. Простой пример: константа

```python
class Ten:
    """Дескриптор, всегда возвращающий 10."""
    def __get__(self, obj, objtype=None):
        return 10


class A:
    x = 5       # обычный атрибут класса
    y = Ten()   # дескриптор


a = A()
print(a.x)  # 5
print(a.y)  # 10 — дескриптор вычисляет значение на лету
```

Обратите внимание: значение `10` **не хранится** ни в классе, ни в экземпляре. Оно **вычисляется** при каждом обращении[reference:22].

## 7.4. Практический пример: валидация

```python
class Positive:
    """Дескриптор: значение должно быть положительным."""

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")
        obj.__dict__[self.name] = value


class Player:
    health = Positive()
    mana = Positive()

    def __init__(self, name, health, mana):
        self.name = name
        self.health = health
        self.mana = mana


p = Player("Hero", 100, 50)
print(p.health)  # 100

try:
    p.health = -10
except ValueError as e:
    print(f"Error: {e}")
```

## 7.5. Связь с `@property`

`@property` — это **встроенный дескриптор**. Когда вы пишете:

```python
@property
def health(self):
    return self._health
```

Python создаёт объект класса `property`, который реализует протокол дескриптора. Свойства — это «дескрипторы для людей».

## 7.6. Применение в игровой разработке

- **Валидация атрибутов:** здоровье, координаты, количество.
- **Ленивая инициализация:** загрузка текстуры при первом обращении.
- **ORM-подобные системы:** доступ к полям базы данных.
- **Сериализация:** автоматическое преобразование типов.

**Для углублённого изучения:**
- [Python docs: Descriptor HowTo](https://docs.python.org/3/howto/descriptor.html)
- [Real Python: Descriptors in Python](https://realpython.com/python-descriptors/)

In [ ]:
class Positive:
    """Дескриптор: значение должно быть положительным."""

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")
        obj.__dict__[self.name] = value


class Player:
    health = Positive()
    mana = Positive()

    def __init__(self, name, health, mana):
        self.name = name
        self.health = health
        self.mana = mana

    def __repr__(self):
        return f"Player({self.name!r}, hp={self.health}, mp={self.mana})"


p = Player("Hero", 100, 50)
print(p)

try:
    p.health = -10
except ValueError as e:
    print(f"Error: {e}")

p.mana = 80
print(p)

---
# 8. Современные практики проектирования интерфейсов

## 8.1. Что такое «интерфейс» в Python

В Python нет ключевого слова `interface` (как в Java или C#). Но **интерфейс** как концепция существует: это **набор методов и атрибутов**, которые класс обязуется предоставить. Интерфейсы позволяют:

- **Документировать контракт** — что класс умеет, а не как он это делает.
- **Изолировать зависимость** — код зависит от интерфейса, а не от реализации.
- **Тестировать** — подменять реальные объекты заглушками.

## 8.2. Три способа описания интерфейса в Python

| Способ | Когда использовать | Пример |
|---|---|---|
| **ABC** | Когда вы владеете иерархией и хотите явный контракт | `class Weapon(ABC)` |
| **Protocol** | Когда важно «что умеет», а не «чем является» | `class Drawable(Protocol)` |
| **Неформальный** | В небольших проектах или для внутренних API | просто методы без наследования |

## 8.3. Правила хорошего интерфейса

1. **Минимализм.** Чем меньше методов, тем проще реализовать интерфейс. Один метод — идеально (`Runnable`, `Drawable`).
2. **Ясность.** Название интерфейса должно отражать его назначение. `Serializable`, `Updatable`, `Renderable`.
3. **Стабильность.** Интерфейс не должен меняться часто. Если меняется — это ломающее изменение.
4. **Документированность.** Каждый метод должен иметь docstring с описанием контракта.
5. **Не наследовать от конкретных классов.** Интерфейс не должен требовать реализации.

## 8.4. Практический пример: система рендеринга

```python
from typing import Protocol


class Renderable(Protocol):
    """Всё, что можно отрисовать на экране."""

    def draw(self, screen) -> None:
        """Нарисовать объект на экране."""
        ...

    @property
    def z_order(self) -> int:
        """Порядок отрисовки (меньше — раньше)."""
        ...


class Renderer:
    """Рендерер, работающий с любыми Renderable."""

    def __init__(self):
        self._objects: list[Renderable] = []

    def add(self, obj: Renderable) -> None:
        self._objects.append(obj)

    def render(self, screen) -> None:
        for obj in sorted(self._objects, key=lambda o: o.z_order):
            obj.draw(screen)
```

**Преимущества:**
- `Renderer` не знает о конкретных классах — только о `Renderable`.
- Легко добавить новый тип объекта — достаточно реализовать `draw()` и `z_order`.
- Легко тестировать — можно подставить Mock-объект с нужными методами.

## 8.5. Комбинирование инструментов

Современный Python-код часто **комбинирует** все инструменты:

- **ABC** — для базовых классов, которые вы контролируете.
- **Protocol** — для внешних зависимостей и гибких контрактов.
- **dataclass** — для компонентов и конфигураций.
- **Mixin** — для переиспользуемых кусочков поведения.
- **`__slots__`** — для оптимизации памяти там, где это важно.

**Для углублённого изучения:**
- [Real Python: Python Interfaces](https://realpython.com/python-interface/)
- [PEP 544 — Protocols](https://peps.python.org/pep-0544/)
- [Real Python: Python's ABC and Protocol](https://realpython.com/python-abstract-base-classes/)

---
# 9. Итоги и литература

## Что мы изучили

| Инструмент | Назначение | Ключевой метод/декоратор |
|---|---|---|
| **ABC** | Явный контракт, запрет инстанцирования | `ABC`, `@abstractmethod` |
| **dataclass** | Автогенерация `__init__`, `__repr__`, `__eq__` | `@dataclass`, `field()`, `frozen` |
| **Protocol** | Структурная типизация без наследования | `Protocol`, `@runtime_checkable` |
| **Mixin** | Переиспользование поведения | `class XMixin`, `super()` |
| **`__slots__`** | Оптимизация памяти | `__slots__ = ('x', 'y')` |
| **Дескрипторы** | Управление доступом к атрибутам | `__get__`, `__set__`, `__set_name__` |

## Практические рекомендации

1. **Используйте ABC** для базовых классов оружия, врагов, способностей.
2. **Используйте dataclass** для компонентов (`Position`, `Health`), конфигураций и событий.
3. **Используйте Protocol** для систем рендеринга, обновления, сохранения — там, где важна гибкость.
4. **Используйте миксины** для добавления логирования, сериализации, подписки на события.
5. **Предпочитайте композицию** наследованию, если поведение можно комбинировать.
6. **Используйте `__slots__`** для снарядов, частиц и других массовых объектов.
7. **Дескрипторы** — для валидации атрибутов и ленивой инициализации.

## Литература

**Базовая:**
- Лутц М. «Изучаем Python», главы 26–27.
- [Python docs: abc](https://docs.python.org/3/library/abc.html)
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)
- [Python docs: typing.Protocol](https://typing.python.org/en/latest/spec/protocol.html)

**Углублённая:**
- Калб И. «Object-Oriented Python» (No Starch, 2022).
- Ramalho L. «Fluent Python», 2-е изд. — главы о протоколах и дескрипторах.
- [Real Python: Abstract Base Classes](https://realpython.com/python-abstract-base-classes/)
- [Real Python: Data Classes](https://realpython.com/python-data-classes/)
- [Real Python: Descriptors](https://realpython.com/python-descriptors/)
- [CodeGym: Composition vs Inheritance](https://codegym.cc/groups/posts/python-composition-vs-inheritance)
- [Python Cookbook: Mixins](https://python-cookbook.readthedocs.io/zh-cn/stable/c08/p18_extending_classes_with_mixins.html)

## Что дальше

В модуле 3 мы изучим принципы SOLID и рефакторинг. Вы научитесь:
- анализировать код на нарушение принципов;
- выделять ответственности;
- применять инверсию зависимостей;
- рефакторить «божественные объекты».

Все инструменты из этого модуля будут активно использоваться.